# 03 — Qwen3-VL-8B Fine-Tuning with Unsloth QLoRA

Fine-tunes Qwen3-VL-8B on financial document extraction using
QLoRA (4-bit quantized LoRA) for efficient training on T4/A100.

**Estimated time**: ~4 hrs on A100, ~12 hrs on T4


In [ ]:
# Cell 1: Load model with Unsloth
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    'Qwen/Qwen3-VL-8B-Instruct',
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)

# Add LoRA adapters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=42,
)

print(f'Trainable params: {model.print_trainable_parameters()}')


In [ ]:
# Cell 2: Load training data
import json
from pathlib import Path
from PIL import Image

TRAINING_DIR = Path('/content/drive/MyDrive/numera-ml/data/training')
train_data = json.loads((TRAINING_DIR / 'train.json').read_text())
val_data = json.loads((TRAINING_DIR / 'val.json').read_text())

print(f'Train: {len(train_data)}, Val: {len(val_data)}')

# Convert to HuggingFace dataset format
from datasets import Dataset

def format_for_training(item):
    img = Image.open(item['image']).convert('RGB')
    user_msg = item['conversations'][0]['content']
    assistant_msg = item['conversations'][1]['content']
    return {
        'image': img,
        'instruction': user_msg,
        'output': assistant_msg,
    }

train_dataset = Dataset.from_list([format_for_training(d) for d in train_data])
val_dataset = Dataset.from_list([format_for_training(d) for d in val_data])
print(f'Datasets ready: train={len(train_dataset)}, val={len(val_dataset)}')


In [ ]:
# Cell 3: Train
from transformers import TrainingArguments
from trl import SFTTrainer
import mlflow

# Configure MLflow tracking
mlflow.set_tracking_uri('file:///content/drive/MyDrive/numera-ml/mlruns')

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/numera-ml/models/checkpoints/qwen3vl',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    save_total_limit=3,
    optim='adamw_8bit',
    seed=42,
    report_to='mlflow',
)

with mlflow.start_run(run_name='qwen3vl-financial-finetune'):
    mlflow.log_params({
        'model': 'Qwen3-VL-8B-Instruct',
        'method': 'QLoRA',
        'lora_r': 16,
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
    })

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=training_args,
        max_seq_length=2048,
    )

    trainer.train()
    print('Training complete! ✅')


In [ ]:
# Cell 4: Save LoRA adapters
ADAPTER_DIR = '/content/drive/MyDrive/numera-ml/models/qwen3vl-financial-lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapters saved to {ADAPTER_DIR}')
